<a href="https://colab.research.google.com/github/miso-20/ESSA/blob/main/ESAA_YB_WEEK_13_2-review4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **수상작 리뷰**

웹 광고 클릭률 예측 AI 경진대회

https://dacon.io/competitions/official/236258/codeshare/10908?page=1&dtype=recent

## **주제**

웹 광고 클릭률 예측

## **데이터**

train.csv
- 시간 순으로 나열된 7일 동안의 웹 광고 클릭 로그
- ID: train 데이터 샘플 고유 ID
- Click: 예측 목표인 클릭 여부
- 0: 클릭하지 않음, 1: 클릭
- F01 ~ F39 : 각 클릭 로그와 연관된 Feature
- 개인정보 보호를 위해 상세 정보는 비식별 처리됨


test.csv
- 시간 순으로 나열된 1일 동안의 웹 광고 클릭 로그
- train 데이터의 다음 날에 해당
- Click이 존재하지 않아, 이를 예측하는 것이 목표
- ID: test 데이터 샘플 고유 ID
- F01 ~ F39 : 각 클릭 로그와 연관된 Feature
- 개인정보 보호를 위해 상세 정보는 비식별 처리됨


sample_submission.csv - 제출 양식
- ID: test 데이터 샘플 고유 ID
- Click: 각 샘플에 대해서 예측한 클릭 확률을 기입하여 제출

## **코드 흐름**


### 1. 데이터 전처리 및 메모리 최적화

- Catboost는 자체적인 범주형 처리 기능을 지원하지만, RAM 사용량을 줄이기 위해 명시적으로 Label Encoding을 수행

- 전처리가 완료된 데이터를 메모리 효율이 좋은 parquet 형태로 저장 후 로드

### 2. 데이터 샘플링 (Data Sampling)

- 전체 데이터 사용 시 RAM 한계로 인한 커널 종료(OOM)가 발생하여 샘플링 진행

- Click=1인 데이터는 전체(약 550만 개)를 사용하고, Click=0인 데이터는 1000만 개만 언더샘플 (Undersampling)하여 학습 데이터 세트 구축

### 3. 모델 학습 (Catboost)

- 메모리 사용량을 줄이고 학습 속도를 높이기 위해 데이터를 Pool 객체로 변환

- 예상치 못한 커널 종료에 대비해 save_snapshot=True 옵션을 적용하고, early_stopping_rounds=500을 주어 효율적으로 학습

### 4. 배치 단위 추론 (Inference)

- 테스트 데이터 추론 시에도 RAM 한계가 발생하여, batch_size=1000 단위로 잘라서(Batch Inference) 예측을 수행하고 결과를 병합함














**주요 코드**

In [ ]:
#  메모리 최적화를 위해 데이터를 Pool 객체로 변환
train_pool = Pool(X_train, label=y_train, cat_features=categorical_feature_index)
validation_pool = Pool(X_test, label=y_test, cat_features=categorical_feature_index)

# CatBoost 모델 정의 (GPU 사용)
model = CatBoostClassifier(
    iterations=7000, learning_rate=0.1, depth=8,
    eval_metric='AUC', random_seed=42, task_type="GPU"
)

# 모델 학습 (early stopping 및 snapshot 저장 적용)
model.fit(
    train_pool, eval_set=validation_pool,
    early_stopping_rounds=500, save_snapshot=True
)

## **새롭게 알게 된 내용 / 어려운 내용 / 배울 점**

- 평소에는 크게 고려하지 않았던 메모리 사용량(RAM)에 대해 깊이 고민할 수 있었다.

- 데이터 타입을 변경(astype)하고, .parquet 확장자를 활용하며 Pool 객체와 배치(Batch) 추론을 적용하는 등 대용량 데이터를 다루는 실질적인 엔지니어링 기술을 배웠다.